# 🏆 통합 벤치마크 V2 (General + Cold-Start)

로컬에서 개선된 **학습 가능 임베딩**과 **인덕티브(Cold-Start) 평가 체계**가 통합된 버전입니다.

In [ ]:
# 셀 1: 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 셀 2: 코드 다운로드 + 데이터 압축 해제
!git clone -b BPR https://github.com/YHFTF/arxiv-conversational-recommender.git /content/arxiv-recsys

import os
os.chdir('/content/arxiv-recsys')
!unzip -q -o /content/drive/MyDrive/colab_data.zip -d /content/arxiv-recsys/

In [ ]:
# 셀 3: 라이브러리 설치
!pip install -q torch-geometric
!pip install -q -r requirements.txt

In [ ]:
# 셀 4: 공통 설정
import torch, json, os, numpy as np; from datetime import datetime
from torch_geometric.nn import GCNConv, SAGEConv; from torch_geometric.nn.conv import LGConv; from torch_geometric.utils import coalesce, degree
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EMBEDDING_DIM, NUM_LAYERS, LEARNING_RATE, NUM_EPOCHS, EVAL_INTERVAL, SEED, TOP_K, EVAL_BATCH_SIZE, TRAIN_RATIO, VAL_RATIO = 128, 2, 0.005, 100, 5, 42, 20, 4096, 0.8, 0.1
PROJECT_ROOT = '/content/arxiv-recsys'; GRAPH_PATH = os.path.join(PROJECT_ROOT, 'subdataset', 'build_hetero_graph_v2.pt'); META_PATH = os.path.join(PROJECT_ROOT, 'output', 'knowledge_meta.json'); MASTER_FILE = os.path.join(PROJECT_ROOT, 'subdataset', 'arxiv_master_final.json'); OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'output', 'benchmark'); os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# 셀 5: 공통 유틸리티 함수 정의 (Transductive + Cold-Start)
# ============================================================

def load_data():
    "데이터 로드 (Emoji 제거 버전)"
    print("[LOAD] 데이터를 로드하는 중...")
    data = torch.load(GRAPH_PATH, weights_only=False).to(DEVICE)
    with open(META_PATH, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    meta_counts = {
        'domains': len(meta['domains']),
        'tasks': len(meta['tasks']),
        'methods': len(meta['methods'])
    }
    return data, meta, meta_counts

def build_knowledge_ids(data, meta):
    "지식 ID 구축"
    print("[INFO] 지식(Knowledge) ID 텐서를 구축하는 중...")
    with open(MASTER_FILE, 'r', encoding='utf-8') as f:
        master_list = json.load(f)
    num_papers = data['paper'].num_nodes
    paper_knowledge_ids = torch.zeros((num_papers, 3), dtype=torch.long)
    for i, item in enumerate(master_list):
        if i >= num_papers: break
        k_dict = item.get('knowledge', {})
        d_val = k_dict.get('domain', None)
        if isinstance(d_val, list) and len(d_val) > 0: d_val = d_val[0]
        t_val = k_dict.get('task', None)
        if isinstance(t_val, list) and len(t_val) > 0: t_val = t_val[0]
        m_val = k_dict.get('method', None)
        if isinstance(m_val, list) and len(m_val) > 0: m_val = m_val[0]
        paper_knowledge_ids[i] = torch.tensor([
            meta['domains'].get(d_val, 0) if d_val else 0,
            meta['tasks'].get(t_val, 0) if t_val else 0,
            meta['methods'].get(m_val, 0) if m_val else 0
        ])
    return paper_knowledge_ids

def evaluate_ranking(out, edges, num_papers, k=TOP_K, batch_size=EVAL_BATCH_SIZE):
    "일반 All-Item Ranking 평가"
    src, pos_dst = edges[0], edges[1]
    paper_embeddings = out[:num_papers]
    total_edges = src.size(0)
    all_hits, all_ndcgs = [], []
    for i in range(0, total_edges, batch_size):
        end = min(i + batch_size, total_edges)
        batch_src, batch_pos_dst = src[i:end], pos_dst[i:end]
        batch_src_embs = out[batch_src]
        all_scores = torch.matmul(batch_src_embs, paper_embeddings.t())
        _, indices = torch.sort(all_scores, dim=1, descending=True)
        rankings = (indices == batch_pos_dst.unsqueeze(1)).nonzero(as_tuple=True)[1]
        hits = (rankings < k).float()
        ndcg = (1.0 / torch.log2(rankings.float() + 2.0))
        ndcg[rankings >= k] = 0.0
        all_hits.append(hits)
        all_ndcgs.append(ndcg)
    return torch.cat(all_hits).mean().item(), torch.cat(all_ndcgs).mean().item()

def evaluate_cold_start(model, test_out, test_edges, num_papers, k=TOP_K, batch_size=EVAL_BATCH_SIZE):
    "신규 노드(Cold-Start) 시뮬레이션 평가"
    src, pos_dst = test_edges[0], test_edges[1]
    paper_embeddings = test_out[:num_papers]
    total_edges = src.size(0)
    all_hits, all_ndcgs = [], []
    for i in range(0, total_edges, batch_size):
        end = min(i + batch_size, total_edges)
        batch_src, batch_pos_dst = src[i:end], pos_dst[i:end]
        if hasattr(model, 'get_cold_start_embeddings'):
            batch_src_embs = model.get_cold_start_embeddings(batch_src)
        else:
            batch_src_embs = torch.zeros((batch_src.size(0), paper_embeddings.size(1)), device=paper_embeddings.device)
        all_scores = torch.matmul(batch_src_embs, paper_embeddings.t())
        _, indices = torch.sort(all_scores, dim=1, descending=True)
        rankings = (indices == batch_pos_dst.unsqueeze(1)).nonzero(as_tuple=True)[1]
        hits = (rankings < k).float()
        ndcg = (1.0 / torch.log2(rankings.float() + 2.0))
        ndcg[rankings >= k] = 0.0
        all_hits.append(hits)
        all_ndcgs.append(ndcg)
    return torch.cat(all_hits).mean().item(), torch.cat(all_ndcgs).mean().item()

def train_and_evaluate(model, train_edges, val_edges, test_edges, num_papers, model_name="Model", save_path=None):
    "통합 학습 루프 (Dual-Phase Metrics)"
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_val_ndcg = -float('inf')
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        out = model(train_edges)
        pos_src, pos_dst = train_edges[0], train_edges[1]
        neg_dst = torch.randint(0, model.total_nodes, (pos_src.size(0),), device=DEVICE)
        pos_scores = (out[pos_src] * out[pos_dst]).sum(dim=-1)
        neg_scores = (out[pos_src] * out[neg_dst]).sum(dim=-1)
        bpr_loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-15).mean()
        bpr_loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_out = model(train_edges)
            v_recall, v_ndcg = evaluate_ranking(val_out, val_edges, num_papers)
            if v_ndcg > best_val_ndcg:
                best_val_ndcg = v_ndcg
                if save_path: torch.save(model.state_dict(), save_path)
        if epoch % EVAL_INTERVAL == 0 or epoch == 1:
            print(f"  [{model_name}] Ep {epoch:3d} | Loss: {bpr_loss.item():.4f} | Val N@{TOP_K}: {v_ndcg:.4f}")
    if save_path and os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, weights_only=True))
    model.eval()
    with torch.no_grad():
        test_out = model(train_edges)
        r, n = evaluate_ranking(test_out, test_edges, num_papers)
        csr, csn = evaluate_cold_start(model, test_out, test_edges, num_papers)
        print(f"  [OK] Test R: {r:.4f} | N: {n:.4f}  [CS] Cold-Start R: {csr:.4f} | N: {csn:.4f}")
    return r, n, csr, csn


In [ ]:
# ============================================================
# 셀 6: 베이스라인 + Ours 모델 클래스 정의 (Cold-Start 지원)
# ============================================================

class BPRMF(nn.Module):
    def __init__(self, total_nodes, initial_features):
        super().__init__()
        self.total_nodes = total_nodes
        self.embedding = nn.Parameter(initial_features.clone())
        self.register_buffer('initial_raw_x', initial_features.clone())
    def forward(self, edge_index=None): return self.embedding
    def get_cold_start_embeddings(self, node_ids): return self.initial_raw_x[node_ids]

class GCNBPR(nn.Module):
    def __init__(self, total_nodes, initial_features, embedding_dim=128, num_layers=2):
        super().__init__()
        self.total_nodes = total_nodes
        self.embedding = nn.Parameter(initial_features.clone())
        self.convs = nn.ModuleList([GCNConv(embedding_dim, embedding_dim) for _ in range(num_layers)])
        self.register_buffer('initial_raw_x', initial_features.clone())
    def forward(self, edge_index):
        x = self.embedding
        xs = [x]
        for conv in self.convs:
            x = torch.relu(conv(x, edge_index))
            xs.append(x)
        return torch.stack(xs, dim=0).mean(dim=0)
    def get_cold_start_embeddings(self, node_ids): return self.initial_raw_x[node_ids]

class GraphSAGEBPR(nn.Module):
    def __init__(self, total_nodes, initial_features, embedding_dim=128, num_layers=2):
        super().__init__()
        self.total_nodes = total_nodes
        self.embedding = nn.Parameter(initial_features.clone())
        self.convs = nn.ModuleList([SAGEConv(embedding_dim, embedding_dim) for _ in range(num_layers)])
        self.register_buffer('initial_raw_x', initial_features.clone())
    def forward(self, edge_index):
        x = self.embedding
        xs = [x]
        for conv in self.convs:
            x = conv(x, edge_index)
            xs.append(x)
        return torch.stack(xs, dim=0).mean(dim=0)
    def get_cold_start_embeddings(self, node_ids): return self.initial_raw_x[node_ids]

class LightGCN(nn.Module):
    def __init__(self, total_nodes, initial_features, embedding_dim=128, num_layers=2):
        super().__init__()
        self.total_nodes = total_nodes
        self.embedding = nn.Parameter(initial_features.clone())
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])
        self.register_buffer('initial_raw_x', initial_features.clone())
    def forward(self, edge_index):
        row, col = edge_index.to(torch.long)
        deg = degree(col, self.total_nodes, dtype=self.embedding.dtype)
        deg_inv_sqrt = deg.pow(-0.5); deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        emb = self.embedding; embs = [emb]
        for conv in self.convs:
            emb = conv(emb, edge_index, edge_weight=norm); embs.append(emb)
        return torch.stack(embs, dim=0).mean(dim=0)
    def get_cold_start_embeddings(self, node_ids): return self.initial_raw_x[node_ids]

class LightGCNKnowledge(nn.Module):
    def __init__(self, total_nodes, num_papers, paper_x, author_x, topic_x,
                 meta_counts, paper_knowledge_ids, embedding_dim=128, num_layers=2):
        super().__init__()
        self.total_nodes = total_nodes; self.num_papers = num_papers
        self.paper_base_x = nn.Parameter(paper_x.clone())
        self.register_buffer('paper_raw_x', paper_x.clone())
        self.register_buffer('paper_knowledge_ids', paper_knowledge_ids.long().to(paper_x.device))
        self.domain_emb = nn.Embedding(meta_counts['domains']+1, embedding_dim, padding_idx=0)
        self.task_emb = nn.Embedding(meta_counts['tasks']+1, embedding_dim, padding_idx=0)
        self.method_emb = nn.Embedding(meta_counts['methods']+1, embedding_dim, padding_idx=0)
        self.author_emb = nn.Parameter(author_x.clone()); self.topic_emb = nn.Parameter(topic_x.clone())
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])
    def _get_combined_embedding(self):
        d_v = self.domain_emb(self.paper_knowledge_ids[:,0])
        t_v = self.task_emb(self.paper_knowledge_ids[:,1])
        m_v = self.method_emb(self.paper_knowledge_ids[:,2])
        dy_p_x = self.paper_base_x + d_v + t_v + m_v
        return torch.cat([dy_p_x, self.author_emb, self.topic_emb], dim=0)
    def forward(self, edge_index):
        row, col = edge_index.to(torch.long)
        deg = degree(col, self.total_nodes, dtype=self.paper_base_x.dtype)
        deg_inv_sqrt = deg.pow(-0.5); deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        emb = self._get_combined_embedding(); embs = [emb]
        for conv in self.convs:
            emb = conv(emb, edge_index, edge_weight=norm); embs.append(emb)
        return torch.stack(embs, dim=0).mean(dim=0)
    def get_cold_start_embeddings(self, node_ids):
        d_v = self.domain_emb(self.paper_knowledge_ids[node_ids,0])
        t_v = self.task_emb(self.paper_knowledge_ids[node_ids,1])
        m_v = self.method_emb(self.paper_knowledge_ids[node_ids,2])
        return self.paper_raw_x[node_ids] + d_v + t_v + m_v


In [ ]:
# 셀 7: 데이터 전처리
data, meta, meta_counts = load_data(); num_papers, num_authors, num_topics, total_nodes = get_graph_info(data); full_edges = build_unified_graph(data); train_edges, val_edges, test_edges = split_edges(full_edges); initial_features = build_initial_features(data); paper_knowledge_ids = build_knowledge_ids(data, meta); paper_x = data['paper'].x; author_x = initial_features[num_papers:num_papers + num_authors]; topic_x = initial_features[num_papers + num_authors:]

In [ ]:
# ============================================================
# 셀 8: 🏆 벤치마크 실행 (Dual-Phase)
# ============================================================

models = [
    {"name": "BPR-MF", "type": "Baseline", "knowledge": "X", "model": BPRMF(total_nodes, initial_features)},
    {"name": "GCN + BPR", "type": "GNN", "knowledge": "X", "model": GCNBPR(total_nodes, initial_features)},
    {"name": "GraphSAGE + BPR", "type": "GNN", "knowledge": "X", "model": GraphSAGEBPR(total_nodes, initial_features)},
    {"name": "LightGCN (BPR)", "type": "GNN", "knowledge": "X", "model": LightGCN(total_nodes, initial_features)},
    {"name": "LightGCN + Knowledge (Ours)", "type": "KG-GNN", "knowledge": "O (LLM)", 
     "model": LightGCNKnowledge(total_nodes, num_papers, paper_x, author_x, topic_x, meta_counts, paper_knowledge_ids)},
]

results = []
for i, m_info in enumerate(models, 1):
    print(f"\n--- [{i}/{len(models)}] {m_info['name']} 학습 시작 ---")
    model = m_info['model'].to(DEVICE)
    save_path = os.path.join(OUTPUT_DIR, f"benchmark_{m_info['name'].replace(' ', '_').lower()}.pt")
    r, n, csr, csn = train_and_evaluate(model, train_edges, val_edges, test_edges, num_papers, model_name=m_info['name'], save_path=save_path)
    results.append({"name": m_info['name'], "type": m_info['type'], "recall": r, "ndcg": n, "cs_recall": csr, "cs_ndcg": csn})
    del model; torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 셀 9: 📊 결과 리포트 (General + Cold-Start)
# ============================================================

print("\n" + "=" * 105)
print("🏆 통합 벤치마크 결과 (Transductive + Cold-Start)")
print(f"   평가 설정: Recall@{TOP_K}, NDCG@{TOP_K} | Seed={SEED} | Epochs={NUM_EPOCHS}")
print("=" * 105)

header = f"| {'Model':<28} | {'Type':<8} | {'Recall':>9} | {'NDCG':>9} | {'CS-Recall':>9} | {'CS-NDCG':>9} |"
print(header); print("|" + "-"*30 + "|" + "-"*10 + "|" + "-"*11 + "|" + "-"*11 + "|" + "-"*11 + "|" + "-"*11 + "|")

best_r = max(r['recall'] for r in results)
best_csn = max(r['cs_ndcg'] for r in results)

for res in results:
    r_str = f"{res['recall']*100:.2f}%" + ("*" if res['recall'] == best_r else "")
    csn_str = f"{res['cs_ndcg']:.4f}" + ("#" if res['cs_ndcg'] == best_csn else "")
    print(f"| {res['name']:<28} | {res['type']:<8} | {r_str:>9} | {res['ndcg']:.4f} | {res['cs_recall']*100:.2f}% | {csn_str:>9} |")

print("=" * 105)
print("  * CS: Cold-Start (신규 논문 시뮬레이션 성능)")

ours = next(r for r in results if 'Ours' in r['name'])
best_bl_cs = max([r['cs_ndcg'] for r in results if 'Ours' not in r['name']])
if best_bl_cs > 0:
    imp = ((ours['cs_ndcg'] - best_bl_cs) / best_bl_cs) * 100
    print(f"\n📈 [Cold-Start] Ours vs 최고 베이스라인 개선율: {imp:+.2f}%")


In [ ]:
# 셀 10: 결과 백업
import shutil; d_bak = '/content/drive/MyDrive/benchmark_results'; os.makedirs(d_bak, exist_ok=True)
for f in os.listdir(OUTPUT_DIR): shutil.copy2(os.path.join(OUTPUT_DIR, f), os.path.join(d_bak, f))
print(f"백업 완료: {d_bak}")